In [2]:
import json
import time
import requests
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np

from bot_template import BaseBot, OrderBook, Order, OrderRequest, OrderResponse, Trade, Side, Product

In [3]:
LONDON_LAT, LONDON_LON = 51.5074, -0.1278

def get_weather(past_steps=96, forecast_steps=96):
    """15-min weather for London. 96 steps = 24 hours.

    Returns DataFrame with: time, temperature, wind_speed, humidity,
    precipitation, cloud_cover, visibility, apparent_temperature.
    """
    variables = "temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,wind_speed_10m,cloud_cover,visibility"
    resp = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": LONDON_LAT, "longitude": LONDON_LON,
        "minutely_15": variables,
        "past_minutely_15": past_steps,
        "forecast_minutely_15": forecast_steps,
        "timezone": "Europe/London",
    })
    resp.raise_for_status()
    m = resp.json()["minutely_15"]
    return pd.DataFrame({
        "time": pd.to_datetime(m["time"]).tz_localize("Europe/London"),
        "temperature": m["temperature_2m"],
        "apparent_temperature": m["apparent_temperature"],
        "humidity": m["relative_humidity_2m"],
        "precipitation": m["precipitation"],
        "wind_speed": m["wind_speed_10m"],
        "cloud_cover": m["cloud_cover"],
        "visibility": m["visibility"],
    })

df_weather = get_weather()
print(f"{len(df_weather)} readings, {df_weather.time.min()} -> {df_weather.time.max()}")
df_weather.tail(5)



192 readings, 2026-02-28 09:45:00+00:00 -> 2026-03-02 09:30:00+00:00


,time,temperature,apparent_temperature,humidity,precipitation,wind_speed,cloud_cover,visibility
187,2026-03-02 08:30:00+00:00,9.7,6.6,74,0.0,13.7,100,16000.0
188,2026-03-02 08:45:00+00:00,9.9,6.7,73,0.0,14.0,100,16060.0
189,2026-03-02 09:00:00+00:00,10.1,6.9,72,0.0,14.4,100,16120.0
190,2026-03-02 09:15:00+00:00,10.4,7.2,71,0.0,14.4,100,16120.0
191,2026-03-02 09:30:00+00:00,10.6,7.5,71,0.0,14.4,100,16120.0


In [4]:
import pandas as pd

# 1. Convert Celsius to Fahrenheit
df_weather['temp_F'] = (df_weather['temperature'] * 9/5) + 32

# 2. Calculate the base metric (temp_F x humidity)
df_weather['wx_metric'] = df_weather['temp_F'] * df_weather['humidity']

# 3. Define the current 12pm-to-12pm session window
# Note: Since we are currently at Feb 28, 2026 ~2:30 PM, the active session is:
session_start = pd.to_datetime("2026-02-28 12:00:00").tz_localize("Europe/London")
session_end   = pd.to_datetime("2026-03-01 12:00:00").tz_localize("Europe/London")

# Filter the DataFrame to only include the current session window
session_df = df_weather[(df_weather['time'] > session_start) & (df_weather['time'] <= session_end)]

# 4. Calculate WX_SPOT (Value exactly at 12 PM at the end of the session)
spot_row = session_df[session_df['time'] == session_end]
wx_spot_fair_value = spot_row['wx_metric'].iloc[0] if not spot_row.empty else None

# 5. Calculate WX_SUM (15-min aggregate / 100)
# Note: Assuming the aggregate is the sum of the wx_metric over the session window. 
wx_sum_fair_value = session_df['wx_metric'].sum() / 100

print(f"Fair Value WX_SPOT: {wx_spot_fair_value}")
print(f"Fair Value WX_SUM:  {wx_sum_fair_value}")

Fair Value WX_SPOT: 4498.0599999999995
Fair Value WX_SUM:  3244.7497999999996


In [5]:
import requests
import pandas as pd
from datetime import datetime, timedelta

def get_london_weather_data():
    # API endpoint and coordinates provided in the rules
    url = "https://api.open-meteo.com/v1/forecast"
    
    # Parameters to get 15-minute data in Fahrenheit for London
    params = {
        "latitude": 51.5074,
        "longitude": -0.1278,
        "minutely_15": "temperature_2m,relative_humidity_2m",
        "temperature_unit": "fahrenheit",
        "timezone": "Europe/London",
        "past_days": 2, # Ensures we capture yesterday (Saturday)
        "forecast_days": 1 # Ensures we capture today (Sunday)
    }
    
    print("Fetching weather data from Open-Meteo...")
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    
    # Load into a pandas DataFrame for easy time filtering
    df = pd.DataFrame({
        "time": pd.to_datetime(data["minutely_15"]["time"]),
        "temperature": data["minutely_15"]["temperature_2m"],
        "humidity": data["minutely_15"]["relative_humidity_2m"]
    })
    
    return df

def calculate_markets(df, target_sunday):
    # Define the settlement window: Saturday 12:00 PM to Sunday 12:00 PM
    sunday_noon = target_sunday.replace(hour=12, minute=0, second=0, microsecond=0)
    saturday_noon = sunday_noon - timedelta(days=1)
    
    # Filter the dataframe for the specific trading window
    window_df = df[(df['time'] >= saturday_noon) & (df['time'] <= sunday_noon)].copy()
    
    if window_df.empty:
        print("Error: No data found for the specified time window.")
        return None, None

    # Apply rounding to temperature as specified in the rules
    window_df['temp_rounded'] = window_df['temperature'].round()
    
    # Calculate T * H for each row
    window_df['T_times_H'] = window_df['temp_rounded'] * window_df['humidity']

    # --- Market 3: WX_SPOT ---
    # Temperature (Fahrenheit, rounded) * humidity at time of settlement (Sunday 12:00 PM)
    spot_row = window_df[window_df['time'] == sunday_noon]
    if not spot_row.empty:
        wx_spot = spot_row.iloc[0]['T_times_H']
    else:
        wx_spot = None
        print("Warning: Could not find exact data for Sunday 12:00 PM.")

    # --- Market 4: WX_SUM ---
    # The sum of (Temperature * Humidity) / 100 over all 15m time intervals
    # Note: Using the exact window from Sat 12:00 to Sun 12:00 (inclusive of boundaries depending on exact exchange logic)
    wx_sum = (window_df['T_times_H'] / 100).sum()

    return wx_spot, wx_sum

if __name__ == "__main__":
    # Fetch the data
    weather_df = get_london_weather_data()
    
    # For the sake of this script, we assume "target_sunday" is the most recent Sunday.
    # If running this live during the competition, today (March 1, 2026) is Sunday.
    today = datetime.now()
    # Find the date of the current/most recent Sunday
    target_sunday = today - timedelta(days=today.weekday() + 1 if today.weekday() != 6 else 0)
    
    wx_spot, wx_sum = calculate_markets(weather_df, target_sunday)
    
    print("\n--- Market Fair Value Estimates ---")
    print(f"Target Settlement Time: {target_sunday.strftime('%Y-%m-%d 12:00:00')}")
    print(f"Market 3 (WX_SPOT): {wx_spot}")
    print(f"Market 4 (WX_SUM):  {wx_sum:.2f}")

Fetching weather data from Open-Meteo...

--- Market Fair Value Estimates ---
Target Settlement Time: 2026-03-01 12:00:00
Market 3 (WX_SPOT): 4539.0
Market 4 (WX_SUM):  3286.93


In [ ]:
Fair Value WX_SPOT: 4498.0599999999995
Fair Value WX_SUM:  3244.7497999999996